# Sync Bedrock Knowledge Base

Trigger an ingestion job on each Bedrock Knowledge Base so it picks up the content just uploaded to S3 by `3_upload_to_s3.ipynb`.

Requires `FIXED_DATA_SOURCE_ID` / `NOCHUNK_DATA_SOURCE_ID` to be set in `backend/.env`, alongside the existing `FIXED_KNOWLEDGE_BASE_ID` / `NOCHUNK_KNOWLEDGE_BASE_ID` (each Bedrock KB's data source ID is visible on the Data source tab of that knowledge base in the AWS console).

> **Note:** this whole notebook (both the code below and the walkthrough here) is AI-generated guidance, not a verified record of what was actually clicked — the knowledge bases for this project were created, configured, and synced manually in the AWS Console.
>
> **1. Create the knowledge base**
> 1. Open **Amazon Bedrock** in the AWS Console → **Knowledge bases** (left sidebar) → **Create knowledge base**.
> 2. Give it a name, and either create a new service role or pick an existing one (needs permissions to read the S3 source, invoke the embedding model, and access the Pinecone secret).
> 3. Choose data source type: **Amazon S3**.
>
> **2. Configure the S3 data source**
> 1. Set the S3 URI to the relevant prefix — `s3://<bucket>/sources_txt/fixed_size/` for the fixed-size KB, or `s3://<bucket>/sources_txt/no_chunking/` for the other.
> 2. Pick a chunking strategy: **Fixed size** with chunk size 500 tokens and 10% overlap for the fixed-size KB, or **No chunking** for the other.
>
> **3. Configure embeddings + vector store (Pinecone)**
> 1. Choose an embedding model (e.g. Titan Text Embeddings V2).
> 2. Before this step, a Pinecone index must already exist (created in the Pinecone console/API, with a dimension matching the embedding model's output size) and its API key must be stored as a secret in **AWS Secrets Manager**.
> 3. In the vector store step, choose **Pinecone**, and provide the Pinecone endpoint URL and the Secrets Manager secret containing the API key.
> 4. Review and click **Create knowledge base** — this provisions the KB and data source, and connects it to the existing Pinecone index.
>
> **4. Sync data from the S3 folder**
> 1. Once created, open the KB's **Data source** tab.
> 2. Select the data source and click **Sync**.
> 3. Wait for the status to go from **Syncing** to **Available** — that means the S3 content has been chunked, embedded, and written into the Pinecone index.
> 4. Repeat steps 1–4 (creation and sync) for the second knowledge base.

In [ ]:
import os
import time

import boto3

from dotenv import load_dotenv

load_dotenv("../backend/.env")

AWS_REGION = os.getenv("AWS_REGION")

FIXED_KNOWLEDGE_BASE_ID = os.getenv("FIXED_KNOWLEDGE_BASE_ID")
FIXED_DATA_SOURCE_ID = os.getenv("FIXED_DATA_SOURCE_ID")

NOCHUNK_KNOWLEDGE_BASE_ID = os.getenv("NOCHUNK_KNOWLEDGE_BASE_ID")
NOCHUNK_DATA_SOURCE_ID = os.getenv("NOCHUNK_DATA_SOURCE_ID")

bedrock_agent = boto3.client("bedrock-agent", region_name = AWS_REGION)


In [ ]:
# Start an ingestion job and return its job id.
def sync_knowledge_base(knowledge_base_id, data_source_id):

    response = bedrock_agent.start_ingestion_job(knowledgeBaseId = knowledge_base_id, dataSourceId = data_source_id)

    job = response["ingestionJob"]

    print(f"Started ingestion job {job['ingestionJobId']} (status: {job['status']})")

    return job["ingestionJobId"]


# Poll an ingestion job until it reaches a terminal status.
def wait_for_ingestion_job(knowledge_base_id, data_source_id, ingestion_job_id, poll_seconds = 10):

    while True:

        response = bedrock_agent.get_ingestion_job(knowledgeBaseId = knowledge_base_id, dataSourceId = data_source_id, ingestionJobId = ingestion_job_id)

        status = response["ingestionJob"]["status"]

        print(f"Status: {status}")

        if status in ("COMPLETE", "FAILED"):
            return status

        time.sleep(poll_seconds)


## Sync the fixed-size chunking knowledge base

In [ ]:
fixed_job_id = sync_knowledge_base(FIXED_KNOWLEDGE_BASE_ID, FIXED_DATA_SOURCE_ID)
wait_for_ingestion_job(FIXED_KNOWLEDGE_BASE_ID, FIXED_DATA_SOURCE_ID, fixed_job_id)


## Sync the no-chunking knowledge base

In [ ]:
nochunk_job_id = sync_knowledge_base(NOCHUNK_KNOWLEDGE_BASE_ID, NOCHUNK_DATA_SOURCE_ID)
wait_for_ingestion_job(NOCHUNK_KNOWLEDGE_BASE_ID, NOCHUNK_DATA_SOURCE_ID, nochunk_job_id)
